<a href="https://colab.research.google.com"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Training YOLOv8 - Deteksi Barang Gudang (38 Kelas)
**Skripsi: Sistem Pendeteksian Barang Gudang Berbasis Computer Vision**

---
## 📋 Daftar Isi
1. [Setup GPU & Install Library](#setup)
2. [Mount Google Drive & Upload Dataset](#dataset)
3. [Persiapan Dataset & Validasi](#validasi)
4. [Training Model YOLOv8](#training)
5. [Evaluasi & Visualisasi Hasil](#evaluasi)
6. [Download Model (best.pt)](#download)

---
### ⚙️ Konfigurasi Dataset
| Parameter | Nilai |
|-----------|-------|
| Total gambar train | 710 |
| Total gambar val | 195 |
| Total class | **38 class** |
| Format label | Bounding Box + Polygon (mixed, didukung YOLOv8) |
| Model dasar | YOLOv8n (Nano) - cepat & ringan |


---
## Sel 1: Cek GPU & Install Library
> **Penting!** Pastikan Colab sudah menggunakan GPU. Klik menu **Runtime → Change runtime type → T4 GPU** sebelum menjalankan notebook ini.

In [ ]:
# ============================================================
# CEK GPU
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU TERDETEKSI! Training akan berjalan cepat.")
    print(result.stdout[:500])
else:
    print("❌ GPU TIDAK TERDETEKSI!")
    print("   → Klik Runtime → Change runtime type → Hardware accelerator: T4 GPU")
    print("   → Lalu restart runtime dan jalankan ulang dari sel ini.")

# ============================================================
# INSTALL ULTRALYTICS (YOLOv8)
# ============================================================
print("\n📦 Menginstall Ultralytics...")
!pip install ultralytics --quiet

import ultralytics
ultralytics.checks()
print("\n✅ Ultralytics siap digunakan!")

---
## Sel 2: Mount Google Drive
> Dataset akan diupload ke Google Drive terlebih dahulu agar tidak perlu upload ulang setiap kali session Colab habis.
>
> **Langkah sebelum menjalankan sel ini:**
> 1. Zip folder `dataset_final` dari laptop Mas Luthfi (klik kanan → Send to → Compressed folder)
> 2. Upload file `dataset_final.zip` ke **Google Drive** (bebas di folder mana saja)
> 3. Salin PATH folder tempat file zip tersimpan (misal: `MyDrive/skripsi/dataset_final.zip`)
> 4. Jalankan sel ini, lalu izinkan akses Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive berhasil terhubung!")

# ============================================================
# KONFIGURASI PATH - SESUAIKAN DENGAN LOKASI FILE ZIP ANDA
# ============================================================

# Ganti path ini sesuai lokasi dataset_final.zip di Google Drive Mas Luthfi
GDRIVE_ZIP_PATH = '/content/drive/MyDrive/dataset_final.zip'   # <-- GANTI INI

# Path tujuan ekstrak di Colab (jangan diubah)
COLAB_DATASET_DIR = '/content/dataset_final'
YAML_PATH = f'{COLAB_DATASET_DIR}/data.yaml'

# Nama run training
RUN_NAME = 'model_gudang_v2'

print(f"\n📂 Path zip    : {GDRIVE_ZIP_PATH}")
print(f"📂 Path dataset: {COLAB_DATASET_DIR}")

# Cek apakah file zip ada
if os.path.exists(GDRIVE_ZIP_PATH):
    print(f"✅ File zip ditemukan!")
else:
    print(f"❌ File zip TIDAK ditemukan di: {GDRIVE_ZIP_PATH}")
    print("   → Pastikan path GDRIVE_ZIP_PATH di atas sudah benar.")

---
## Sel 3: Ekstrak Dataset

In [ ]:
import zipfile
import shutil
import time

# Hapus folder lama jika ada (agar ekstraksi bersih)
if os.path.exists(COLAB_DATASET_DIR):
    print(f"🗑️  Menghapus folder lama: {COLAB_DATASET_DIR}")
    shutil.rmtree(COLAB_DATASET_DIR)

print(f"📦 Mengekstrak dataset dari: {GDRIVE_ZIP_PATH}")
print("    (Proses ini mungkin memakan beberapa menit untuk dataset besar...)")

start = time.time()
with zipfile.ZipFile(GDRIVE_ZIP_PATH, 'r') as zip_ref:
    for member in zip_ref.infolist():
        member.filename = member.filename.replace('\\', '/')
        zip_ref.extract(member, '/content/')
elapsed = time.time() - start

print(f"\n✅ Dataset berhasil diekstrak dalam {elapsed:.1f} detik!")

# Verifikasi struktur folder
print("\n📂 Struktur folder dataset:")
for split in ['train', 'val']:
    img_dir = os.path.join(COLAB_DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(COLAB_DATASET_DIR, split, 'labels')
    if os.path.exists(img_dir):
        n_img = len(os.listdir(img_dir))
        n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
        print(f"  {split}/images : {n_img} gambar")
        print(f"  {split}/labels : {n_lbl} file label")
    else:
        print(f"  ❌ Folder {split} TIDAK ditemukan!")

---
## Sel 4: Update data.yaml untuk Google Colab
> File `data.yaml` dari laptop Mas Luthfi menggunakan path lokal (`path: .`). Di Colab, path-nya harus diubah menjadi path absolut Colab agar YOLOv8 bisa menemukan gambar-gambarnya.

In [ ]:
import yaml

# Baca data.yaml yang ada
with open(YAML_PATH, 'r') as f:
    data = yaml.safe_load(f)

# Update path ke path absolut Colab
data['path']  = COLAB_DATASET_DIR
data['train'] = 'train/images'
data['val']   = 'val/images'

# Simpan kembali
with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print("✅ data.yaml berhasil diupdate!")
print(f"   path  : {data['path']}")
print(f"   train : {data['train']}")
print(f"   val   : {data['val']}")
print(f"   nc    : {data.get('nc', 'tidak ada')} class")
print(f"\n📋 Daftar {data.get('nc')} Class:")
names = data.get('names', {})
if isinstance(names, list):
    names = {i: v for i, v in enumerate(names)}
for idx in sorted(names.keys()):
    print(f"  [{idx:>2}] {names[idx]}")

---
## Sel 5: 🏋️ TRAINING MODEL YOLOv8
> Ini adalah sel utama yang menjalankan proses pelatihan (training). 
>
> **Penjelasan Parameter Penting:**
> | Parameter | Nilai | Penjelasan |
> |-----------|-------|------------|
> | `model` | `yolov8n.pt` | Model dasar (nano = paling ringan, cocok untuk Colab gratis) |
> | `epochs` | `100` | Jumlah putaran belajar. Lebih tinggi = lebih pintar (tapi lama) |
> | `imgsz` | `640` | Ukuran gambar saat training (standar YOLOv8) |
> | `batch` | `16` | Jumlah gambar per batch. Turunkan ke 8 jika Colab RAM habis |
> | `patience` | `30` | Berhenti otomatis jika tidak ada kemajuan selama 30 epoch |
> | `optimizer` | `AdamW` | Optimizer terbaik untuk dataset kecil-menengah |
> | `lr0` | `0.001` | Learning rate awal |
> | `augment` | `True` | Data augmentation otomatis (biar model lebih tahan banting) |

In [ ]:
from ultralytics import YOLO
import torch

print(f"🖥️  Device: {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (lambat!)'}")
print(f"📂 Dataset YAML: {YAML_PATH}")
print(f"🏷️  Nama Run    : {RUN_NAME}")
print("\n" + "="*60)
print("  MEMULAI TRAINING... Jangan tutup tab ini!")
print("  Estimasi waktu: 20-40 menit di GPU T4 Colab")
print("="*60 + "\n")

# Load model dasar YOLOv8 nano (pre-trained pada ImageNet)
model = YOLO('yolov8n.pt')

# ============================================================
# JALANKAN TRAINING
# ============================================================
results = model.train(
    data      = YAML_PATH,      # Path ke data.yaml
    epochs    = 100,            # Jumlah epoch
    imgsz     = 640,            # Ukuran gambar
    batch     = 16,             # Batch size (turunkan ke 8 jika OOM)
    patience  = 30,             # Early stopping
    optimizer = 'AdamW',        # Optimizer
    lr0       = 0.001,          # Learning rate awal
    lrf       = 0.01,           # Learning rate akhir (fraction)
    augment   = True,           # Augmentasi otomatis
    mosaic    = 1.0,            # Mosaic augmentation (bagus untuk multi-class)
    mixup     = 0.1,            # Mixup augmentation
    degrees   = 10.0,           # Rotasi gambar (max 10 derajat)
    fliplr    = 0.5,            # Flip horizontal 50%
    hsv_h     = 0.015,          # Variasi Hue
    hsv_s     = 0.7,            # Variasi Saturation
    hsv_v     = 0.4,            # Variasi Value/Brightness
    project   = 'runs/detect',  # Folder output
    name      = RUN_NAME,       # Nama subfolder output
    save      = True,           # Simpan checkpoint
    save_period = 10,           # Simpan checkpoint tiap 10 epoch
    plots     = True,           # Generate grafik training otomatis
    verbose   = True,           # Tampilkan log detail
)

print("\n" + "="*60)
print("  ✅ TRAINING SELESAI!")
print(f"  Model terbaik tersimpan di: runs/detect/{RUN_NAME}/weights/best.pt")
print("="*60)

---
## Sel 6: 📊 Evaluasi Model (Metrik Akurasi)
> Setelah training selesai, jalankan sel ini untuk melihat nilai akurasi model (mAP50, Precision, Recall).
>
> **Penjelasan Metrik:**
> - **Precision**: Dari semua deteksi model, berapa % yang benar?
> - **Recall**: Dari semua objek yang ada, berapa % yang berhasil dideteksi?
> - **mAP50**: Mean Average Precision pada IoU 50% — metrik utama YOLOv8. Semakin tinggi semakin bagus (target: > 0.70)
> - **mAP50-95**: mAP yang lebih ketat, dihitung di berbagai threshold IoU.

In [ ]:
from ultralytics import YOLO

# Load model terbaik hasil training
best_model_path = f'runs/detect/{RUN_NAME}/weights/best.pt'
model_best = YOLO(best_model_path)

# Jalankan validasi
print("🔍 Menjalankan evaluasi model pada data validasi...")
metrics = model_best.val(data=YAML_PATH, imgsz=640, batch=16, verbose=True)

print("\n" + "="*60)
print("  📊 HASIL EVALUASI MODEL")
print("="*60)
print(f"  Precision (P)  : {metrics.box.mp:.4f}  ({metrics.box.mp*100:.2f}%)")
print(f"  Recall    (R)  : {metrics.box.mr:.4f}  ({metrics.box.mr*100:.2f}%)")
print(f"  mAP@50         : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.2f}%)")
print(f"  mAP@50-95      : {metrics.box.map:.4f}  ({metrics.box.map*100:.2f}%)")
print("="*60)

if metrics.box.map50 >= 0.70:
    print("  ✅ Akurasi BAGUS! Model siap digunakan.")
elif metrics.box.map50 >= 0.50:
    print("  ⚠️  Akurasi CUKUP. Pertimbangkan tambah epoch atau data.")
else:
    print("  ❌ Akurasi RENDAH. Perlu lebih banyak data atau epoch.")

---
## Sel 7: 🖼️ Visualisasi Grafik Training

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

result_dir = f'runs/detect/{RUN_NAME}'

# Daftar gambar grafik yang di-generate otomatis oleh YOLOv8
plot_files = {
    'results.png'       : 'Grafik Training (Loss & Metrik per Epoch)',
    'confusion_matrix.png': 'Confusion Matrix',
    'val_batch0_pred.jpg' : 'Contoh Prediksi Batch Validasi',
    'PR_curve.png'      : 'Precision-Recall Curve',
    'F1_curve.png'      : 'F1 Confidence Curve',
}

for filename, title in plot_files.items():
    path = os.path.join(result_dir, filename)
    if os.path.exists(path):
        print(f"\n📊 {title}")
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f"   (File {filename} tidak ditemukan, mungkin belum digenerate)")

---
## Sel 8: 💾 Simpan Model ke Google Drive & Download
> **Sangat penting!** Session Google Colab gratis bisa berakhir kapan saja. Jalankan sel ini segera setelah training selesai untuk menyimpan `best.pt` ke Google Drive supaya tidak hilang!

In [ ]:
import shutil
from google.colab import files
import os

best_pt_path = f'runs/detect/{RUN_NAME}/weights/best.pt'
last_pt_path = f'runs/detect/{RUN_NAME}/weights/last.pt'

# ============================================================
# 1. SIMPAN KE GOOGLE DRIVE (backup aman)
# ============================================================
# Ganti path tujuan di Google Drive sesuai keinginan
GDRIVE_SAVE_DIR = '/content/drive/MyDrive/model_hasil_training'   # <-- GANTI JIKA PERLU

os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)

if os.path.exists(best_pt_path):
    dst = os.path.join(GDRIVE_SAVE_DIR, 'best.pt')
    shutil.copy2(best_pt_path, dst)
    size_mb = os.path.getsize(dst) / 1024 / 1024
    print(f"✅ best.pt berhasil disimpan ke Google Drive!")
    print(f"   Path   : {dst}")
    print(f"   Ukuran : {size_mb:.2f} MB")
else:
    print(f"❌ best.pt tidak ditemukan di: {best_pt_path}")

if os.path.exists(last_pt_path):
    dst_last = os.path.join(GDRIVE_SAVE_DIR, 'last.pt')
    shutil.copy2(last_pt_path, dst_last)
    print(f"✅ last.pt berhasil disimpan ke Google Drive!")

# Simpan juga folder hasil training lengkap (grafik, dll)
results_zip_dst = os.path.join(GDRIVE_SAVE_DIR, f'{RUN_NAME}_results.zip')
shutil.make_archive(
    base_name = results_zip_dst.replace('.zip', ''),
    format    = 'zip',
    root_dir  = f'runs/detect/{RUN_NAME}'
)
print(f"✅ Hasil training lengkap (grafik, dll) tersimpan ke Google Drive!")
print(f"   Path: {results_zip_dst}")

print("\n" + "="*60)
print("  Semua file sudah aman di Google Drive!")
print("  Sekarang jalankan sel berikutnya untuk download ke laptop.")
print("="*60)

---
## Sel 9: ⬇️ Download best.pt ke Laptop

In [ ]:
from google.colab import files

best_pt_path = f'runs/detect/{RUN_NAME}/weights/best.pt'

print(f"⬇️  Mendownload {best_pt_path} ke laptop...")
print("   (Dialog download akan muncul di browser Mas Luthfi)")
files.download(best_pt_path)
print("✅ Download selesai!")
print("\n📌 Langkah selanjutnya:")
print("   1. Pindahkan file best.pt yang baru ke folder project laptop Mas Luthfi")
print("   2. Ganti file best.pt yang lama dengan yang baru")
print("   3. Jalankan python app.py untuk menguji model baru!")

---
## Sel 10 (OPSIONAL): 🔄 Lanjutkan Training dari Checkpoint
> Jalankan sel ini **HANYA JIKA** training sebelumnya terhenti di tengah jalan (misal karena session Colab habis) dan Mas Luthfi ingin melanjutkan dari epoch terakhir tanpa mengulang dari awal.

In [ ]:
# OPSIONAL: Lanjutkan training dari checkpoint last.pt
# Jalankan sel ini hanya jika training terhenti di tengah jalan

from ultralytics import YOLO

# Load dari Google Drive (jika session Colab baru)
last_pt_from_drive = os.path.join(GDRIVE_SAVE_DIR, 'last.pt')

if os.path.exists(last_pt_from_drive):
    print(f"✅ Menemukan checkpoint: {last_pt_from_drive}")
    model_resume = YOLO(last_pt_from_drive)
    results = model_resume.train(
        data    = YAML_PATH,
        resume  = True,       # <-- Kunci: lanjutkan dari epoch terakhir
        epochs  = 100,
        project = 'runs/detect',
        name    = RUN_NAME,
    )
    print("✅ Training lanjutan selesai!")
else:
    print(f"❌ File last.pt tidak ditemukan di Google Drive: {last_pt_from_drive}")
    print("   Pastikan Sel 8 sudah pernah dijalankan untuk menyimpan checkpoint ke Drive.")

---
## 📖 Catatan Penting

| # | Hal | Penjelasan |
|---|-----|------------|
| 1 | **Session Timeout** | Google Colab gratis akan otomatis disconnect setelah ~12 jam atau jika browser ditutup. Selalu simpan ke Drive dulu! |
| 2 | **OOM (Out of Memory)** | Jika muncul error CUDA OOM, turunkan `batch` dari `16` ke `8` di Sel 5. |
| 3 | **mAP rendah** | Coba tambah data augmentasi atau perbanyak gambar di class yang sedikit. |
| 4 | **best.pt vs last.pt** | Selalu gunakan `best.pt` untuk deployment. `last.pt` hanya untuk resume training. |
| 5 | **Format label campuran** | Dataset ini menggunakan Bounding Box (data lama) + Polygon (data baru dari Roboflow). YOLOv8 mendukung keduanya. |

---
*Notebook ini dibuat untuk skripsi Computer Vision Mas Luthfi & Lisna* 🎓